In [3]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

df = pd.read_pickle("../data/processed/movies_features.pkl")

threshold = df["vote_count"].quantile(0.75)
df["high_engagement"] = (df["vote_count"] >= threshold).astype(int)

mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(df["genres"])
genre_cols = [f"genre_{g}" for g in mlb.classes_]

genre_df = pd.DataFrame(genre_matrix, columns=genre_cols, index=df.index)
df = pd.concat([df, genre_df], axis=1)
df

,movie_id,title,overview,release_date,runtime,original_language,genres,keywords,budget,revenue,...,genre_History,genre_Horror,genre_Music,genre_Mystery,genre_Romance,genre_Science Fiction,genre_TV Movie,genre_Thriller,genre_War,genre_Western
0,1101412,Fall 2: Deadpoint,On a perilous climb across Thailand's Mount Kw...,2026-09-01,98.0,en,[Thriller],"[sequel, rock climbing, sport climbing, climb,...",3000000.0,9256541.0,...,0,0,0,0,0,0,0,1,0,0
1,8392,My Neighbor Totoro,Two sisters move to the country with their fat...,1988-04-16,86.0,ja,"[Fantasy, Animation, Family]","[mother, sibling relationship, village, leave,...",3700000.0,41000000.0,...,0,0,0,0,0,0,0,0,0,0
2,5879,In the Realm of the Senses,"A passionate telling of the story of Sada Abe,...",1976-09-15,102.0,ja,"[Drama, Romance]","[japan, jealousy, geisha, brothel, eroticism, ...",NaN,7650000.0,...,0,0,0,0,1,0,0,0,0,0
3,491418,Instant Family,"When Pete and Ellie decide to start a family, ...",2018-11-13,118.0,en,"[Comedy, Drama]","[court case, social worker, adoption, based on...",48000000.0,121000000.0,...,0,0,0,0,0,0,0,0,0,0
4,1100099,We Live in Time,An up-and-coming chef and a recent divorcée fi...,2024-10-10,108.0,en,"[Romance, Drama]","[marriage, father, family relationships, roman...",20000000.0,37182814.0,...,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1714,10515,Castle in the Sky,A young boy and a girl with a magic crystal mu...,1986-08-02,125.0,ja,"[Adventure, Fantasy, Animation, Action, Family]","[army, flying, magic, mine, castle, lost civil...",3000000.0,5228752.0,...,0,0,0,0,0,0,0,0,0,0
1715,22832,Ninja Assassin,"Ninja Assassin follows Raizo, one of the deadl...",2009-09-29,99.0,en,"[Action, Adventure, Thriller]","[martial arts, assassin, assassination, ninja ...",40000000.0,61601280.0,...,0,0,0,0,0,0,0,1,0,0
1716,28,Apocalypse Now,"At the height of the Vietnam war, Captain Benj...",1979-05-19,147.0,en,"[Drama, War]","[guerrilla warfare, vietnam war, journalist, m...",31500000.0,150000000.0,...,0,0,0,0,0,0,0,0,1,0
1717,541671,Ballerina,Taking place during the events of John Wick: C...,2025-06-04,125.0,en,"[Action, Thriller, Crime]","[assassin, tattoo, grenade, austria, training,...",90000000.0,137408107.0,...,0,0,0,0,0,0,0,1,0,0


In [2]:
df.to_pickle("../data/processed/movies_final.pkl")
print("saved")

saved


In [3]:
from sklearn.model_selection import train_test_split

numeric_features = ["runtime", "budget", "revenue", "popularity", "n_genres", "n_keywords", "overview_length"]
categorical_features = ["original_language", "runtime_category"]

X = df[numeric_features + categorical_features + genre_cols  + ["overview_clean"]]
y = df["high_engagement"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

((1375, 29), (344, 29))

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
    ("genre", "passthrough", genre_cols),
    # ("text", TfidfVectorizer(max_features=3000, ngram_range=(1, 2), stop_words="english"), "overview_clean"),
])

In [5]:
from sklearn.linear_model import LogisticRegression

log_reg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])

log_reg_pipeline.fit(X_train, y_train)
print("Trained")

Trained


In [6]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

y_pred = log_reg_pipeline.predict(X_test)
y_proba = log_reg_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.85      0.93      0.89       258
           1       0.70      0.50      0.59        86

    accuracy                           0.82       344
   macro avg       0.78      0.72      0.74       344
weighted avg       0.81      0.82      0.81       344

ROC-AUC: 0.8664593473949883
Confusion matrix:
 [[240  18]
 [ 43  43]]


In [7]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")),
])

rf_pipeline.fit(X_train, y_train)

y_pred_rf = rf_pipeline.predict(X_test)
y_proba_rf = rf_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.92      0.88      0.90       258
           1       0.68      0.77      0.72        86

    accuracy                           0.85       344
   macro avg       0.80      0.82      0.81       344
weighted avg       0.86      0.85      0.85       344

ROC-AUC: 0.9078555976203354
Confusion matrix:
 [[227  31]
 [ 20  66]]


In [8]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

svm_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", random_state=42, max_iter=5000)
    )),
])

svm_pipeline.fit(X_train, y_train)

y_pred_svm = svm_pipeline.predict(X_test)
y_proba_svm = svm_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_svm))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_svm))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_svm))

              precision    recall  f1-score   support

           0       0.85      0.94      0.89       258
           1       0.74      0.49      0.59        86

    accuracy                           0.83       344
   macro avg       0.79      0.72      0.74       344
weighted avg       0.82      0.83      0.82       344

ROC-AUC: 0.8664593473949883
Confusion matrix:
 [[243  15]
 [ 44  42]]


In [9]:
from sklearn.model_selection import StratifiedKFold, cross_validate

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_validate(
    rf_pipeline, X, y, cv=skf,
    scoring=["accuracy", "precision", "recall", "f1", "roc_auc"]
)

for metric in ["test_accuracy", "test_precision", "test_recall", "test_f1", "test_roc_auc"]:
    print(metric, "mean:", round(scores[metric].mean(), 3), "std:", round(scores[metric].std(), 3))

test_accuracy mean: 0.851 std: 0.012
test_precision mean: 0.7 std: 0.03
test_recall mean: 0.712 std: 0.055
test_f1 mean: 0.704 std: 0.029
test_roc_auc mean: 0.901 std: 0.01


In [11]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [None, 10, 20],
    "classifier__min_samples_leaf": [1, 2, 4],
}

grid_search = GridSearchCV(
    rf_pipeline, param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="f1",
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)   # train split only

print("Best params:", grid_search.best_params_)
print("Best F1 (cross-val):", grid_search.best_score_)

Best params: {'classifier__max_depth': 10, 'classifier__min_samples_leaf': 2, 'classifier__n_estimators': 100}
Best F1 (cross-val): 0.6985314869328838


In [ ]:
best_rf_pipeline = grid_search.best_estimator_

y_pred_tuned = best_rf_pipeline.predict(X_test)
y_proba_tuned = best_rf_pipeline.predict_proba(X_test)[:, 1]

print("--- Default Random Forest (before) ---")
print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf))

print("\n--- Tuned Random Forest (after) ---")
print(classification_report(y_test, y_pred_tuned))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_tuned))

--- Default Random Forest (before) ---
              precision    recall  f1-score   support

           0       0.92      0.88      0.90       258
           1       0.68      0.77      0.72        86

    accuracy                           0.85       344
   macro avg       0.80      0.82      0.81       344
weighted avg       0.86      0.85      0.85       344

ROC-AUC: 0.9078555976203354

--- Tuned Random Forest (after) ---
              precision    recall  f1-score   support

           0       0.93      0.86      0.89       258
           1       0.66      0.80      0.72        86

    accuracy                           0.85       344
   macro avg       0.79      0.83      0.81       344
weighted avg       0.86      0.85      0.85       344

ROC-AUC: 0.9018388318009735


In [ ]:
import joblib

joblib.dump(best_rf_pipeline, "../models/classifier_rf.pkl")
print("saved")

saved
